In [2]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv
from urllib.parse import quote

class NaverSearchAPI:
    def __init__(self):
        # .env 파일에서 환경변수 로드
        load_dotenv()
        
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        # API 키 확인
        if not self.client_id or not self.client_secret:
            raise ValueError("API 키가 설정되지 않았습니다. .env 파일을 확인하세요.")
    
    def get_search_count(self, keyword, search_type="blog"):
        """네이버 검색 API로 검색 결과 수 가져오기"""
        
        # API URL 설정
        api_urls = {
            "blog": "https://openapi.naver.com/v1/search/blog.json",
            "news": "https://openapi.naver.com/v1/search/news.json",
            "cafe": "https://openapi.naver.com/v1/search/cafearticle.json",
            "web": "https://openapi.naver.com/v1/search/webkr.json"
        }
        
        url = api_urls.get(search_type, api_urls["blog"])
        
        # 헤더 설정
        headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret
        }
        
        # 파라미터 설정
        params = {
            'query': keyword,
            'display': 1,  # 1개만 가져와서 total만 확인
            'start': 1
        }
        
        try:
            response = requests.get(url, headers=headers, params=params)
            
            if response.status_code == 200:
                data = response.json()
                return data.get('total', 0)
            else:
                print(f"API 오류 ({search_type}): {response.status_code} - {keyword}")
                return 0
                
        except Exception as e:
            print(f"API 호출 오류: {e} - {keyword}")
            return 0
    
    def test_api_connection(self):
        """API 연결 테스트"""
        test_keyword = "김치찌개"
        test_result = self.get_search_count(test_keyword)
        
        if test_result > 0:
            print(f"API 연결 테스트 성공: '{test_keyword}' 블로그 검색 결과 {test_result:,}개")
            return True
        else:
            print("API 연결 테스트 실패. .env 파일의 API 키를 확인하세요.")
            return False
    
    def collect_search_counts(self, keywords):
        """모든 키워드의 검색 결과 수 수집"""
        
        results = []
        total_keywords = len(keywords)
        
        print(f"총 {total_keywords}개 키워드 검색 시작")
        print("-" * 50)
        
        for i, keyword in enumerate(keywords, 1):
            print(f"[{i}/{total_keywords}] {keyword}")
            
            # 각 검색 타입별로 결과 수 가져오기
            blog_count = self.get_search_count(keyword, "blog")
            time.sleep(0.1)
            
            news_count = self.get_search_count(keyword, "news")
            time.sleep(0.1)
            
            cafe_count = self.get_search_count(keyword, "cafe")
            time.sleep(0.1)
            
            web_count = self.get_search_count(keyword, "web")
            time.sleep(0.1)
            
            # 결과 저장
            total_count = blog_count + news_count + cafe_count + web_count
            
            results.append({
                '키워드': keyword,
                '블로그_검색수': blog_count,
                '뉴스_검색수': news_count,
                '카페_검색수': cafe_count,
                '웹_검색수': web_count,
                '총합': total_count
            })
            
            print(f"  블로그: {blog_count:,}, 뉴스: {news_count:,}, 카페: {cafe_count:,}, 웹: {web_count:,}")
            print(f"  총합: {total_count:,}")
            
            # API 호출 제한을 위한 대기
            time.sleep(0.5)
        
        return pd.DataFrame(results)

def load_keywords_from_csv():
    """하드코딩된 카테고리 및 키워드 로딩"""
    try:
        print("카테고리 포함 키워드 로딩 중...")

        keywords = [
            # 햄버거 카테고리
            "햄버거", "버거킹", "쉑쉑버거", "롯데리아",
            # 샌드위치 카테고리
            "샌드위치", "에그드롭", "서브웨이", "홍루이젠",
            # 디저트 카테고리
            "디저트", "노티드", "몽슈슈", "투썸플레이스"
        ]

        print(f"총 {len(keywords)}개 키워드 로딩 완료")
        return keywords

    except Exception as e:
        print(f"키워드 로딩 오류: {e}")
        return None

def process_all_keywords(keywords):
    """전체 키워드 처리"""
    print(f"\n전체 {len(keywords)}개 키워드를 처리합니다.")
    return keywords

def save_results_to_excel(results_df, original_keywords):
    """결과를 엑셀 파일로 저장 - 원래 키워드 순서 유지"""
    
    # 키워드 순서 유지용 정렬 인덱스 생성
    keyword_order = {k: i for i, k in enumerate(original_keywords)}
    results_df['순서'] = results_df['키워드'].map(keyword_order)
    results_df = results_df.sort_values('순서').drop(columns=['순서'])

    # 엑셀 파일로 저장
    output_filename = "네이버_검색_절대숫자.xlsx"
    results_df.to_excel(output_filename, index=False)

    print("\n" + "=" * 50)
    print("수집 완료!")
    print(f"파일 저장: {output_filename}")
    print(f"총 처리된 키워드: {len(results_df)}개")

    # 상위 10개 키워드 출력 (총합 기준)
    print("\n상위 10개 키워드:")
    print("-" * 50)
    top_10 = results_df.sort_values('총합', ascending=False).head(10)
    for _, row in top_10.iterrows():
        print(f"{row['키워드']}: {row['총합']:,}개")

    # 통계 정보
    print(f"\n통계 정보:")
    print(f"평균 검색 결과 수: {results_df['총합'].mean():,.0f}개")
    print(f"최대 검색 결과 수: {results_df['총합'].max():,}개")
    print(f"최소 검색 결과 수: {results_df['총합'].min():,}개")

    return results_df

def create_env_file_template():
    """환경변수 파일 템플릿 생성"""
    env_template = """# 네이버 개발자센터에서 발급받은 API 키를 입력하세요
# https://developers.naver.com/apps/#/register

Client_ID=your_client_id_here
Client_Secret=your_client_secret_here
"""
    
    with open('.env', 'w', encoding='utf-8') as f:
        f.write(env_template)
    
    print(".env 파일 템플릿을 생성했습니다.")
    print("파일을 열어서 API 키를 입력한 후 다시 실행하세요.")

def main():
    print("네이버 검색 API를 사용한 전체 키워드 검색량 조사")
    print("=" * 60)
    
    # .env 파일 확인
    if not os.path.exists('.env'):
        print("오류: .env 파일이 없습니다.")
        print("현재 폴더에 .env 파일이 있는지 확인하세요.")
        return
    
    # API 클래스 초기화
    try:
        api = NaverSearchAPI()
    except ValueError as e:
        print(f"오류: {e}")
        print("\n.env 파일을 확인하세요:")
        print("Client_ID=your_client_id")
        print("Client_Secret=your_client_secret")
        return
    
    # API 연결 테스트
    print("\nAPI 연결 테스트 중...")
    if not api.test_api_connection():
        return
    
    # 키워드 로드
    keywords = load_keywords_from_csv()
    if keywords is None:
        return
    
    # 전체 키워드 처리
    selected_keywords = process_all_keywords(keywords)
    
    # 예상 소요 시간 계산
    estimated_time = len(selected_keywords) * 0.5 / 60  # 키워드당 0.5초 * 분 변환
    print(f"예상 소요 시간: 약 {estimated_time:.1f}분")
    
    # 처리 시작 확인
    start_confirm = input("\n처리를 시작하시겠습니까? (y/n): ").strip().lower()
    if start_confirm != 'y':
        print("처리를 중단합니다.")
        return
    
    # 검색 결과 수 수집
    try:
        print(f"\n전체 키워드 검색량 조사 시작...")
        results_df = api.collect_search_counts(selected_keywords)
        
        # 결과 저장 및 출력
        save_results_to_excel(results_df, selected_keywords)
        
    except Exception as e:
        print(f"처리 중 오류 발생: {e}")
        return

if __name__ == "__main__":
    main()

네이버 검색 API를 사용한 전체 키워드 검색량 조사

API 연결 테스트 중...
API 연결 테스트 성공: '김치찌개' 블로그 검색 결과 3,840,012개
카테고리 포함 키워드 로딩 중...
총 12개 키워드 로딩 완료

전체 12개 키워드를 처리합니다.
예상 소요 시간: 약 0.1분

처리를 시작하시겠습니까? (y/n): y

전체 키워드 검색량 조사 시작...
총 12개 키워드 검색 시작
--------------------------------------------------
[1/12] 햄버거
  블로그: 4,123,926, 뉴스: 206,314, 카페: 1,436,059, 웹: 8,146,163
  총합: 13,912,462
[2/12] 버거킹
  블로그: 860,653, 뉴스: 45,395, 카페: 521,263, 웹: 4,519,714
  총합: 5,947,025
[3/12] 쉑쉑버거
  블로그: 211,655, 뉴스: 2,605, 카페: 127,375, 웹: 79,446
  총합: 421,081
[4/12] 롯데리아
  블로그: 1,128,294, 뉴스: 91,029, 카페: 717,435, 웹: 5,034,685
  총합: 6,971,443
[5/12] 샌드위치
  블로그: 7,090,306, 뉴스: 250,922, 카페: 2,222,365, 웹: 8,221,542
  총합: 17,785,135
[6/12] 에그드롭
  블로그: 22,072, 뉴스: 2,446, 카페: 4,337, 웹: 308,247
  총합: 337,102
[7/12] 서브웨이
  블로그: 646,136, 뉴스: 5,959, 카페: 294,977, 웹: 1,107,237
  총합: 2,054,309
[8/12] 홍루이젠
  블로그: 53,682, 뉴스: 1,092, 카페: 21,652, 웹: 120,127
  총합: 196,553
[9/12] 디저트
  블로그: 17,777,239, 뉴스: 455,352, 카페: 1,839,772, 웹: 19,462,303
  총합: 3